# One Schedule, Two Compositions: Tuning Through a Hub-Delivered Adapter

The [interp-engine adapter](https://github.com/interpretune/it-interp-engine-adapter) is not installed with
Interpretune: it is fetched from the Hub at a pinned revision, trusted explicitly and registered, after which the
compositions `(core, interp_engine)` and `(lightning, interp_engine)` exist. Its compositions declare the same tunable
view as `core`, so a finetuning-scheduler schedule written by address runs on them unchanged. This notebook shows that
with one schedule run twice, once on `(core,)` and once on `(core, interp_engine)`, and checks that the two runs agree
tensor for tensor:

| Stage | Address | What it tunes |
|---|---|---|
| 0 | `role:intervention:steer` | a steering vector at `point:hook_resid_post & layers:8`, alone |
| 1 | `layers:-2:` | the model's last two blocks, beside the steering vector |
| 2 | `role:latent:sae` | a published SAE, promoted from `artifact` to `latent:sae`, with the blocks refrozen |

Stage 0 is interpretability-parameter tuning (a trainable intervention declared through the module's
`register_tunable_modules()`), stage 1 tunes the model's own weights by layer, and stage 2 is latent-space tuning (a
latent model attached to the composition, selected by role). No entry names a parameter, and none names the adapter.

The run is GPT-2 on RTE through Interpretune's core loop, small enough for a CPU.

Requirements: `uv pip install -e ".[examples,tune]" --group git-deps` (CPU torch is fine; the `examples` extra
installs interp-engine).

> **Until the `speediedan/rte` and `speediedan/it-interp-engine-adapter` components are public, this notebook needs a
> Hugging Face token with access to them** (set `HF_TOKEN`). No GPU and no license acceptance are needed.

In [ ]:
# Parameters - These will be injected by papermill during parameterized test runs
LIMIT_TRAIN_BATCHES = 8  # batches per epoch
LIMIT_VAL_BATCHES = 4
SEED = 0
REVISION = (
    None  # rte component snapshot; None resolves the recorded pin (pass a local-publish rev to validate unmerged configs)
)
# The adapter, pinned: the revision you read is the revision that executes.
ADAPTER_REPO = "speediedan/it-interp-engine-adapter"
ADAPTER_REVISION = "40ba5de18993791a28f95356d7c8f254719c2995"

In [ ]:
# @title Imports { display-mode: "form" }
import importlib
import os
from collections import Counter

import torch

import interpretune as it
from interpretune.config import SessionRunnerCfg
from interpretune.harness.notebook_ui import display_notebook_table
from interpretune.runners.core import SessionRunner
from interpretune.tuning.core_loop import FTSCoreCallback
from interpretune.tuning.interventions import OpIntervention, attach_intervention
from interpretune.tuning.latents import LatentSite, attach_latent_model

## 1. Fetch and Register the Adapter

Three steps, kept separate because they are three different decisions: **fetch** (the only networked step, at a pinned
revision), **trust** (running code someone else published, which this notebook opts into for this process only) and
**register** (adding `interp_engine` to this process's adapters). The
[hub-delivered adapter notebook](../interp_engine_example/interp_engine_hub_adapter.ipynb) walks through each one; here
they are one cell. Before it runs, `it.Adapter.interp_engine` does not exist.

In [ ]:
# @title 1: Fetch, trust and register { display-mode: "form" }
if REVISION is None:  # a validation run passes a local-publish REVISION and needs none of these steps
    it.hub.pull("speediedan/rte")  # caches the component's manifest and entrypoint; executes nothing
manifest, resolved_commit = it.hub.pull(ADAPTER_REPO, revision=ADAPTER_REVISION)
os.environ["IT_TRUST_REMOTE_CODE"] = "1"  # opt in to running both components' entrypoints, for this session only
if REVISION is None:
    it.hub.pull("speediedan/rte", "rte_ft.gpt2.core")
load = it.hub.load_hub_adapter(ADAPTER_REPO)
assert [m.name for m in load.members] == ["interp_engine"], load
print(f"registered {[m.name for m in load.members]} from {ADAPTER_REPO} @ {resolved_commit[:12]}")

## 2. One Module, Either Composition

The session is the `rte_ft.gpt2.core` configuration of the `speediedan/rte` component. One module class serves both
compositions: in `register_tunable_modules()`, which runs once the model exists and before its view is built, it
attaches

- **`steer`**, a steering vector (`OpIntervention("add", 768)`) at `point:hook_resid_post & layers:8`, whose operand
  starts at zero, where `add` is the identity. Its tensor's role is `intervention:steer`;
- **`sae`**, a published sparse autoencoder (OpenAI's 32k-latent residual SAE for GPT-2 small's layer 1, loaded through
  sae_lens) spliced in at `point:hook_resid_post & layers:1` with the `replace` op, so the model runs on the SAE's
  reconstruction. It is attached `frozen=True`, so its tensors are `artifact` until a stage promotes them.

The sites are venngram addresses, resolved against whichever composition the module is built in. The only difference
between the two sessions is the composition itself: `adapter_ctx`, and the adapter's own configuration
(`interp_engine_cfg`) for `(core, interp_engine)`.

The engine itself declares `INTERVENTION` and `CAPTURE` and refuses latent-model handles in its analysis interventions.
That does not limit tuning: an attached latent model is spliced into the module by Interpretune, which owns the
module's tunable view on every composition.

In [ ]:
# @title 2: The session, built for either composition { display-mode: "form" }
D_MODEL = 768
SAE_RELEASE, SAE_ID = "gpt2-small-resid-post-v5-32k", "blocks.1.hook_resid_post"
COMPOSITIONS = {"core": (it.Adapter.core,), "core + interp_engine": (it.Adapter.core, it.Adapter.interp_engine)}
# the adapter's configuration class, at the stable alias its registration gives it (as its CLI configurations name it)
interp_engine_config = importlib.import_module("it_hub_adapters.speediedan__it_interp_engine_adapter").InterpEngineConfig


def load_sae() -> torch.nn.Module:
    from sae_lens import SAE

    loaded = SAE.from_pretrained(SAE_RELEASE, SAE_ID, device="cpu")
    return loaded[0] if isinstance(loaded, tuple) else loaded


def build_session(adapter_ctx: tuple) -> it.ITSession:
    itdm_cfg, it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", "rte_ft.gpt2.core", revision=REVISION)
    if it.Adapter.interp_engine in adapter_ctx:
        it_cfg.interp_engine_cfg = interp_engine_config()
        it_cfg.auto_comp_cfg.target_adapters = "interp_engine"

    class TunedModule(m_cls):
        def register_tunable_modules(self) -> None:
            super().register_tunable_modules()
            attach_intervention(self, "steer", OpIntervention("add", D_MODEL), "point:hook_resid_post & layers:8")
            attach_latent_model(
                self, "sae", load_sae(), LatentSite("point:hook_resid_post & layers:1", op="replace"), frozen=True
            )

    session = it.ITSession(
        it.ITSessionConfig(
            adapter_ctx=adapter_ctx,
            datamodule_cfg=itdm_cfg,
            module_cfg=it_cfg,
            datamodule_cls=dm_cls,
            module_cls=TunedModule,
        )
    )
    it.it_init(**session)
    return session

## 3. The Schedule, Run on Each Composition

```python
{
    0: {"params": ["role:intervention:steer"], "max_transition_epoch": 1},
    1: {"params": ["layers:-2:"], "max_transition_epoch": 2},
    2: {"params": ["role:latent:sae"], "promote": {"role:artifact": "latent:sae"}, "refreeze": ["layers:-2:"]},
}
```

`layers:-2:` names the last two blocks of whatever model the view describes, so the same entry thaws GPT-2's blocks 10
and 11 here and a deeper model's last two elsewhere. Stage 2 promotes the SAE's `artifact` tensors to `latent:sae`,
thaws them, and holds the two blocks where stage 1 left them (`refreeze`). `promote` and `refreeze` run on the core
loop (`FTSCoreCallback`). The steering vector, once thawed, keeps training in the later stages.

Each run starts from the same seed. A small core-loop callback records, at the end of each stage's epoch, which
tensors could take a gradient and the epoch's metrics; it runs before the tuning scheduler's own callback, so it reads
each stage before the next one starts.

In [ ]:
# @title 3: Run the schedule on both compositions { display-mode: "form" }
SCHEDULE = {
    0: {"params": ["role:intervention:steer"], "max_transition_epoch": 1},
    1: {"params": ["layers:-2:"], "max_transition_epoch": 2},
    2: {"params": ["role:latent:sae"], "promote": {"role:artifact": "latent:sae"}, "refreeze": ["layers:-2:"]},
}


def run(adapter_ctx: tuple) -> dict:
    torch.manual_seed(SEED)
    session = build_session(adapter_ctx)
    module = session.module
    initial = {n: p.detach().clone() for n, p in module.named_parameters()}
    composed = getattr(module, "_composed_classes", ())
    result = {"roles": dict(module.tunable_view.roles()), "stages": {}, "composed": [c.__name__ for c in composed]}

    class RecordStages:
        def on_train_epoch_end(self, ctx, metrics) -> None:
            result["stages"][fts.tuning_scheduler.curr_depth] = {
                "thawed": sorted(n for n, p in module.named_parameters() if p.requires_grad),
                "metrics": {k: float(v) for k, v in (metrics or {}).items()},
            }

    fts = FTSCoreCallback(SCHEDULE)
    SessionRunner(
        run_cfg=SessionRunnerCfg(
            it_session=session,
            max_epochs=len(SCHEDULE),
            limit_train_batches=LIMIT_TRAIN_BATCHES,
            limit_val_batches=LIMIT_VAL_BATCHES,
            callbacks=[RecordStages(), fts],  # the recorder first, so it reads each stage before a transition
        )
    ).train()
    assert fts.tuning_scheduler.curr_depth == len(SCHEDULE) - 1, "every stage must run"
    result["final"] = {n: p.detach().clone() for n, p in module.named_parameters()}
    result["changed"] = sorted(n for n, p in result["final"].items() if not torch.equal(initial[n], p))
    return result


runs = {name: run(ctx) for name, ctx in COMPOSITIONS.items()}
display_notebook_table(
    [[name, ", ".join(r["composed"])] for name, r in runs.items()],
    ["Composition", "Module composed from"],
    title="The module each run trained",
)
assert runs["core"]["composed"] != runs["core + interp_engine"]["composed"], "the second run composes the adapter"

## 4. What Each Stage Tuned, and Whether the Compositions Agree

The first table reads each stage on each composition: how many tensors of each kind could take a gradient. The second
compares the two runs: the tensors each stage thawed, the metrics each epoch reported, and the trained values.

In [ ]:
# @title 4: Each stage, and the parity check { display-mode: "form" }
def kind(name: str) -> str:
    if name.startswith("interventions.steer."):
        return "steer"
    if name.startswith("latent_models.sae."):
        return "sae"
    return "base"


display_notebook_table(
    [
        [name, depth, *(Counter(kind(n) for n in stage["thawed"]).get(k, 0) for k in ("steer", "base", "sae"))]
        for name, r in runs.items()
        for depth, stage in sorted(r["stages"].items())
    ],
    ["Composition", "Stage", "steer tensors", "base tensors", "sae tensors"],
    title="Tensors thawed in each stage",
    caption="Role counts in the view: "
    + ", ".join(f"{role} {n}" for role, n in sorted(Counter(runs["core"]["roles"].values()).items()))
    + " (every other tensor is `base`)",
)

core, engine = runs["core"], runs["core + interp_engine"]
max_diff = max((core["final"][n] - engine["final"][n]).abs().max().item() for n in core["final"])
display_notebook_table(
    [
        ["Same tensors thawed in each stage", core["stages"].keys() == engine["stages"].keys()
         and all(core["stages"][d]["thawed"] == engine["stages"][d]["thawed"] for d in core["stages"])],
        ["Same metrics each epoch", all(core["stages"][d]["metrics"] == engine["stages"][d]["metrics"]
                                        for d in core["stages"])],
        ["Same tensors changed by the run", core["changed"] == engine["changed"]],
        ["Largest difference between trained values", f"{max_diff:.3g}"],
    ],
    ["Check", "core vs core + interp_engine"],
    title="Parity between the two compositions",
    caption="val_loss by stage, core: "
    + ", ".join(f"{s['metrics'].get('val_loss', float('nan')):.3f}" for _, s in sorted(core["stages"].items())),
)

for name, r in runs.items():
    stages = r["stages"]
    assert {kind(n) for n in stages[0]["thawed"]} == {"steer"}, f"{name}: stage 0 thaws the steering vector alone"
    blocks = [n for n in stages[1]["thawed"] if kind(n) == "base"]
    assert blocks and all(n.split(".h.")[1].split(".")[0] in ("10", "11") for n in blocks), (
        f"{name}: `layers:-2:` thaws GPT-2's last two blocks"
    )
    assert not any(kind(n) == "base" for n in stages[2]["thawed"]), f"{name}: stage 2 refreezes the blocks"
    assert any(kind(n) == "sae" for n in stages[2]["thawed"]), f"{name}: stage 2 trains the promoted SAE"
    assert {kind(n) for n in r["changed"]} == {"steer", "base", "sae"}, f"{name}: every stage changed what it tuned"
assert core["changed"] == engine["changed"] and max_diff == 0.0, "the two compositions trained identically"

**Reading the tables.**

- **Stage 0** thaws one tensor, the steering vector, on both compositions. **Stage 1** adds the last two blocks'
  tensors, found by `layers:-2:` in each composition's view. **Stage 2** promotes the SAE, thaws its tensors and
  refreezes the blocks.
- **The two runs agree exactly**: the same tensors in every stage, the same metrics every epoch, and trained values
  that do not differ. The interp-engine adapter wraps the model Interpretune already loaded rather than reloading it,
  and its compositions declare the same view, so the schedule had nothing to translate.

A schedule written by address is a property of the experiment rather than of the backend: the same file runs on a
composition delivered from the Hub as on the ones Interpretune bundles. The adapter's own configurations
(`examples/rte_ft` in its repository) run the `speediedan/rte` component's full schedule the same way from the command
line, on the core loop or on Lightning.